In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PrimeLogistics-ONU") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/opt/spark/spark-warehouse") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 14:28:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [2]:
from pyspark.sql import functions as F

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/opt/spark/datos/commodity_trade_statistics_data.csv")

print("Total filas:", df.count())
print("\nEsquema:")
df.printSchema()
df.show(3, truncate=False)

Total filas: 8226597

Esquema:
root
 |-- country_or_area: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- comm_code: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- flow: string (nullable = true)
 |-- trade_usd: long (nullable = true)
 |-- weight_kg: long (nullable = true)
 |-- quantity_name: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- category: string (nullable = true)

+---------------+----+---------+---------------------------------------+------+---------+---------+---------------+--------+---------------+
|country_or_area|year|comm_code|commodity                              |flow  |trade_usd|weight_kg|quantity_name  |quantity|category       |
+---------------+----+---------+---------------------------------------+------+---------+---------+---------------+--------+---------------+
|Afghanistan    |2016|010410   |Sheep, live                            |Export|6088     |2339     |Number of items|51.0    |01_live_ani

In [3]:
# ── 1. Limpiar datos nulos y tipos incorrectos ──────────────────────────────
df_clean = df \
    .filter(F.col("trade_usd").isNotNull()) \
    .filter(F.col("weight_kg").isNotNull()) \
    .filter(F.col("year").isNotNull()) \
    .filter(F.col("flow").isin("Export", "Import")) \
    .withColumn("trade_usd", F.col("trade_usd").cast("long")) \
    .withColumn("weight_kg", F.col("weight_kg").cast("long")) \
    .withColumn("year", F.col("year").cast("integer"))

print("Filas después de limpiar:", df_clean.count())

# ── 2. ¿Qué se mueve? — Ranking de mercancías por valor ────────────────────
df_que_se_mueve = df_clean \
    .groupBy("category", "commodity", "flow") \
    .agg(
        F.sum("trade_usd").alias("total_usd"),
        F.sum("weight_kg").alias("total_kg"),
        F.count("*").alias("num_transacciones")
    ) \
    .withColumn("valor_por_kg", F.round(F.col("total_usd") / F.col("total_kg"), 2)) \
    .orderBy(F.desc("total_usd"))

df_que_se_mueve.write.mode("overwrite").parquet("/opt/spark/spark-warehouse/que_se_mueve")
print("✅ Tabla 'que_se_mueve' guardada")

# ── 3. ¿Quién lo mueve? — Ranking de países ────────────────────────────────
df_quien_mueve = df_clean \
    .groupBy("country_or_area", "flow", "year") \
    .agg(
        F.sum("trade_usd").alias("total_usd"),
        F.sum("weight_kg").alias("total_kg"),
        F.count("*").alias("num_transacciones")
    ) \
    .orderBy(F.desc("total_usd"))

df_quien_mueve.write.mode("overwrite").parquet("/opt/spark/spark-warehouse/quien_mueve")
print("✅ Tabla 'quien_mueve' guardada")

# ── 4. ¿Cuánto vale y cuánto pesa? — Rentabilidad ─────────────────────────
df_rentabilidad = df_clean \
    .groupBy("category", "year") \
    .agg(
        F.sum("trade_usd").alias("total_usd"),
        F.sum("weight_kg").alias("total_kg")
    ) \
    .withColumn("usd_por_kg", F.round(F.col("total_usd") / F.col("total_kg"), 4)) \
    .orderBy(F.desc("usd_por_kg"))

df_rentabilidad.write.mode("overwrite").parquet("/opt/spark/spark-warehouse/rentabilidad")
print("✅ Tabla 'rentabilidad' guardada")

# ── 5. Desequilibrios comerciales (para mapa de calor Superset) ────────────
df_desequilibrio = df_clean \
    .groupBy("country_or_area", "year") \
    .pivot("flow", ["Export", "Import"]) \
    .agg(F.sum("trade_usd")) \
    .fillna(0) \
    .withColumn("balance_usd", F.col("Export") - F.col("Import")) \
    .orderBy(F.desc("balance_usd"))

df_desequilibrio.write.mode("overwrite").parquet("/opt/spark/spark-warehouse/desequilibrio")
print("✅ Tabla 'desequilibrio' guardada")

print("\n🎉 Pipeline completo. 4 tablas Parquet listas para Superset.")

Filas después de limpiar: 7592012


✅ Tabla 'que_se_mueve' guardada


✅ Tabla 'quien_mueve' guardada


✅ Tabla 'rentabilidad' guardada


✅ Tabla 'desequilibrio' guardada

🎉 Pipeline completo. 4 tablas Parquet listas para Superset.


In [4]:
# Exportar tablas optimizadas como CSV para Superset
df_que_se_mueve = spark.read.parquet("/opt/spark/spark-warehouse/que_se_mueve")
df_que_se_mueve.toPandas().to_csv("/opt/spark/spark-warehouse/que_se_mueve.csv", index=False)

df_quien_mueve = spark.read.parquet("/opt/spark/spark-warehouse/quien_mueve")
df_quien_mueve.toPandas().to_csv("/opt/spark/spark-warehouse/quien_mueve.csv", index=False)

df_rentabilidad = spark.read.parquet("/opt/spark/spark-warehouse/rentabilidad")
df_rentabilidad.toPandas().to_csv("/opt/spark/spark-warehouse/rentabilidad.csv", index=False)

df_desequilibrio = spark.read.parquet("/opt/spark/spark-warehouse/desequilibrio")
df_desequilibrio.toPandas().to_csv("/opt/spark/spark-warehouse/desequilibrio.csv", index=False)

print("✅ CSVs exportados")

ImportError: Pandas >= 1.0.5 must be installed; however, it was not found.

In [6]:
import subprocess
subprocess.run(["pip", "install", "pandas", "--quiet"], check=True)
print("✅ pandas instalado")

✅ pandas instalado


In [7]:
# Exportar tablas optimizadas como CSV para Superset
df_que_se_mueve = spark.read.parquet("/opt/spark/spark-warehouse/que_se_mueve")
df_que_se_mueve.toPandas().to_csv("/opt/spark/spark-warehouse/que_se_mueve.csv", index=False)

df_quien_mueve = spark.read.parquet("/opt/spark/spark-warehouse/quien_mueve")
df_quien_mueve.toPandas().to_csv("/opt/spark/spark-warehouse/quien_mueve.csv", index=False)

df_rentabilidad = spark.read.parquet("/opt/spark/spark-warehouse/rentabilidad")
df_rentabilidad.toPandas().to_csv("/opt/spark/spark-warehouse/rentabilidad.csv", index=False)

df_desequilibrio = spark.read.parquet("/opt/spark/spark-warehouse/desequilibrio")
df_desequilibrio.toPandas().to_csv("/opt/spark/spark-warehouse/desequilibrio.csv", index=False)

print("✅ CSVs exportados")

✅ CSVs exportados


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/opt/spark/datos/prime_logistics.sqlite")

spark.read.parquet("/opt/spark/spark-warehouse/que_se_mueve").toPandas().to_sql("que_se_mueve", conn, if_exists="replace", index=False)
print("✅ que_se_mueve")

spark.read.parquet("/opt/spark/spark-warehouse/quien_mueve").toPandas().to_sql("quien_mueve", conn, if_exists="replace", index=False)
print("✅ quien_mueve")

spark.read.parquet("/opt/spark/spark-warehouse/rentabilidad").toPandas().to_sql("rentabilidad", conn, if_exists="replace", index=False)
print("✅ rentabilidad")

spark.read.parquet("/opt/spark/spark-warehouse/desequilibrio").toPandas().to_sql("desequilibrio", conn, if_exists="replace", index=False)
print("✅ desequilibrio")

conn.close()
print("🎉 SQLite listo")

✅ que_se_mueve
✅ quien_mueve
✅ rentabilidad
✅ desequilibrio
🎉 SQLite listo
